In [1]:
from ultralytics import YOLO

model = YOLO('runs/segment/v2_1280_m/weights/best.pt')

In [2]:
model.predict('C:/Users/Moha/Pictures/1.jpg',imgsz = 1280,save = True)

image 1/1 C:\Users\Moha\Pictures\1.jpg: 864x1280 2 vans, 8 trucks, 3 buss, 23.4ms


Speed: 26.6ms preprocess, 23.4ms inference, 17.2ms postprocess per image at shape (1, 3, 864, 1280)


Results saved to C:\Users\Moha\Desktop\FULL-PROJECTS\SPEEDESTIM\CODE\runs\segment\predict-7


[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 depth: None
 keypoints: None
 masks: ultralytics.engine.results.Masks object
 names: {0: 'car', 1: 'van', 2: 'truck', 3: 'bus'}
 obb: None
 orig_img: array([[[189, 192, 207],
         [202, 208, 221],
         [216, 225, 238],
         ...,
         [ 52,  47,  44],
         [ 53,  48,  45],
         [ 51,  46,  43]],
 
        [[190, 191, 205],
         [192, 195, 209],
         [206, 215, 225],
         ...,
         [ 37,  33,  32],
         [ 40,  36,  35],
         [ 52,  48,  47]],
 
        [[206, 206, 218],
         [224, 226, 236],
         [223, 230, 239],
         ...,
         [ 43,  42,  44],
         [ 26,  25,  27],
         [ 30,  29,  31]],
 
        ...,
 
        [[224, 215, 206],
         [224, 215, 206],
         [225, 216, 207],
         ...,
         [ 94,  89,  88],
         [ 92,  87,  86],
         [ 93,  88,  87]],
 
        [[224, 215, 206],
       

In [3]:
import pathlib as pl
import json
import math

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

TRACKS = pl.Path("../speed_data/tracks_v2")
WINDOW = 30  # 30 frames before and after, so 61 frames


def load_split(split):
    data = []
    for file in sorted((TRACKS / split).glob("*.json")):
        with open(file) as f:
            data.append(json.load(f))
    return data

In [4]:
def build_cars(data):
    # group the rows by car, the id starts again in every clip so the key is (clip, id)
    cars = {}
    for clip_i, clip in enumerate(data):
        w = clip["width"]
        h = clip["height"]
        cam = clip["camera"]
        camera = [cam["height_above_road_m"], cam["pitch"], cam["fov"]]

        for frame_i, frame in enumerate(clip["frames"]):
            for vehicle in frame:
                key = (clip_i, vehicle["id"])
                if key not in cars:
                    cars[key] = {"class": vehicle["class"], "camera": camera, "frames": {}, "kmh": {}}

                x1, y1, x2, y2 = vehicle["box"]
                lx, ly = vehicle["lowest"]
                # 1 if the box touches the border of the image (car is cut off)
                edge = float(x1 <= 0 or y1 <= 0 or x2 >= w - 1 or y2 >= h - 1)

                # road_m and kmh are not used as input
                cars[key]["frames"][frame_i] = [lx / w, ly / h, x1 / w, y1 / h, x2 / w, y2 / h,
                                                math.log(vehicle["pixels"]), edge]
                cars[key]["kmh"][frame_i] = vehicle["kmh"]
    return cars

In [5]:
def build_windows(cars, step=5, min_present=10):
    # one window of 61 frames around every 5th frame of a car
    # token = present, lowest x, lowest y, box (4), log pixels, edge, dx, dy, time
    X, M, Y, CLS, CAM = [], [], [], [], []
    for car in cars.values():
        seen = car["frames"]
        for t in sorted(seen)[::step]:
            lx_t = seen[t][0]
            ly_t = seen[t][1]
            tokens, present, speeds = [], [], []
            for k in range(-WINDOW, WINDOW + 1):
                if t + k in seen:
                    f = seen[t + k]
                    tokens.append([1.0] + f + [f[0] - lx_t, f[1] - ly_t, k / WINDOW])
                    present.append(True)
                    speeds.append(car["kmh"][t + k])
                else:
                    tokens.append([0.0] * 11 + [k / WINDOW])
                    present.append(False)
                    speeds.append(0.0)
            # skip windows where the car is almost never visible
            if sum(present) < min_present:
                continue
            X.append(tokens)
            M.append(present)
            Y.append(speeds)
            CLS.append(car["class"])
            CAM.append(car["camera"])

    X = torch.tensor(X, dtype=torch.float32)
    M = torch.tensor(M)
    Y = torch.tensor(Y, dtype=torch.float32)
    CLS = torch.tensor(CLS)
    CAM = torch.tensor(CAM, dtype=torch.float32)
    return X, M, Y, CLS, CAM


def preprocess(split, step=5):
    return build_windows(build_cars(load_split(split)), step=step)

In [6]:
X_train, M_train, Y_train, CLS_train, CAM_train = preprocess("train")
X_val, M_val, Y_val, CLS_val, CAM_val = preprocess("val")

print("train:", X_train.shape, "val:", X_val.shape)

train: torch.Size([143938, 61, 12]) val: torch.Size([22766, 61, 12])


In [7]:
# Normalise with numbers from the train set only (log pixels, dx, dy and the camera)
SCALE_COLUMNS = [7, 9, 10]


def fit_normaliser(X, M, CAM):
    real = X[M]  # only frames where the car is visible
    return {"x_mean": real[:, SCALE_COLUMNS].mean(0), "x_std": real[:, SCALE_COLUMNS].std(0),
            "cam_mean": CAM.mean(0), "cam_std": CAM.std(0)}


def normalise(X, M, CAM, stats):
    X = X.clone()
    X[:, :, SCALE_COLUMNS] = (X[:, :, SCALE_COLUMNS] - stats["x_mean"]) / stats["x_std"]
    X[~M] = 0.0
    X[:, :, 11] = torch.arange(-WINDOW, WINDOW + 1) / WINDOW  # put the time back on missing frames
    CAM = (CAM - stats["cam_mean"]) / stats["cam_std"]
    return X, CAM


stats = fit_normaliser(X_train, M_train, CAM_train)
X_train_n, CAM_train_n = normalise(X_train, M_train, CAM_train, stats)
X_val_n, CAM_val_n = normalise(X_val, M_val, CAM_val, stats)
stats

{'x_mean': tensor([ 7.2206e+00, -5.7715e-05, -1.0738e-04]),
 'x_std': tensor([1.7639, 0.0605, 0.0692]),
 'cam_mean': tensor([ 11.3767, -11.0677,  53.6952]),
 'cam_std': tensor([ 4.1114,  5.1954, 14.9522])}

In [8]:
class SpeedTransformer(nn.Module):
    # every frame is a token, class and camera get added to every token
    def __init__(self, n_features=12, d_model=128, n_heads=4, n_layers=4, dropout=0.1):
        super().__init__()
        self.token_in = nn.Linear(n_features, d_model)
        self.position = nn.Embedding(2 * WINDOW + 1, d_model)
        self.class_emb = nn.Embedding(4, d_model)
        self.camera_in = nn.Linear(3, d_model)
        layer = nn.TransformerEncoderLayer(d_model, n_heads, dim_feedforward=4 * d_model, dropout=dropout,
                                           batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, 1)

    def forward(self, x, present, cls, cam):
        positions = torch.arange(x.shape[1], device=x.device)
        h = self.token_in(x) + self.position(positions)
        h = h + (self.class_emb(cls) + self.camera_in(cam))[:, None, :]
        # the mask is True for frames to ignore, so it's the opposite of present
        h = self.encoder(h, src_key_padding_mask=~present)
        return self.head(self.norm(h)).squeeze(-1)  # speed / 100 for every frame


device = "cuda" if torch.cuda.is_available() else "cpu"
model = SpeedTransformer().to(device)
print(device, sum(p.numel() for p in model.parameters()), "parameters")

cuda 803969 parameters


In [9]:
EPOCHS = 30
BATCH = 512
PATIENCE = 8

# frames far from t count less in the loss (1 at t, 0.5 at the ends)
weight_per_frame = (1 - torch.arange(-WINDOW, WINDOW + 1).abs() / (2 * WINDOW)).to(device)


def masked_loss(pred, target, present):
    loss = nn.functional.huber_loss(pred, target, reduction="none", delta=0.1)
    weight = present * weight_per_frame
    return (loss * weight).sum() / weight.sum()


train_loader = DataLoader(TensorDataset(X_train_n, M_train, Y_train / 100, CLS_train, CAM_train_n),
                          batch_size=BATCH, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_n, M_val, Y_val / 100, CLS_val, CAM_val_n), batch_size=2048)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=1e-3, total_steps=EPOCHS * len(train_loader))


def evaluate(loader):
    # error in km/h at frame t only, that's the speed we use live
    model.eval()
    errors = []
    with torch.no_grad():
        for x, m, y, c, cam in loader:
            pred = model(x.to(device), m.to(device), c.to(device), cam.to(device))
            errors.append((pred[:, WINDOW].cpu() - y[:, WINDOW]).abs() * 100)
    return torch.cat(errors).mean().item()


best = 1000
bad_epochs = 0
for epoch in range(EPOCHS):
    model.train()
    total = 0
    for x, m, y, c, cam in train_loader:
        x, m, y, c, cam = x.to(device), m.to(device), y.to(device), c.to(device), cam.to(device)
        loss = masked_loss(model(x, m, c, cam), y, m)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total += loss.item()

    val_mae = evaluate(val_loader)
    print("epoch", epoch + 1, "train loss", round(total / len(train_loader), 4), "val MAE", round(val_mae, 2), "km/h")

    if val_mae < best:
        best = val_mae
        bad_epochs = 0
        torch.save({"model": model.state_dict(), "stats": stats}, "speed_transformer.pt")
    else:
        bad_epochs += 1
        # only stop in the second half, the first half is jumpy because of the high learning rate
        if bad_epochs >= PATIENCE and epoch >= EPOCHS // 2:
            print("stopped early")
            break

print("best val MAE", round(best, 2), "km/h")

epoch 1 train loss 0.0148 val MAE 20.31 km/h


epoch 2 train loss 0.0118 val MAE 18.19 km/h


epoch 3 train loss 0.0087 val MAE 13.89 km/h


epoch 4 train loss 0.0063 val MAE 12.57 km/h


epoch 5 train loss 0.0052 val MAE 11.56 km/h


epoch 6 train loss 0.0045 val MAE 12.38 km/h


... (20 outputs left out)

epoch 27 train loss 0.0006 val MAE 7.46 km/h


epoch 28 train loss 0.0006 val MAE 7.37 km/h


epoch 29 train loss 0.0006 val MAE 7.32 km/h


epoch 30 train loss 0.0006 val MAE 7.35 km/h
best val MAE 7.32 km/h


In [10]:
# load the best model and check it per class, against just guessing the average speed
model.load_state_dict(torch.load("speed_transformer.pt")["model"])
model.eval()

preds = []
with torch.no_grad():
    for x, m, y, c, cam in val_loader:
        preds.append(model(x.to(device), m.to(device), c.to(device), cam.to(device))[:, WINDOW].cpu() * 100)
pred = torch.cat(preds)
true = Y_val[:, WINDOW]
error = (pred - true).abs()

print("transformer MAE", round(error.mean().item(), 2), "km/h")
average = Y_train[:, WINDOW].mean()
print("guessing the average", round(average.item(), 1), "km/h every time:", round((average - true).abs().mean().item(), 2), "km/h")
for i, name in enumerate(["car", "van", "truck", "bus"]):
    print(name, round(error[CLS_val == i].mean().item(), 2), "km/h")

transformer MAE 7.32 km/h
guessing the average 67.9 km/h every time: 21.0 km/h
car 7.19 km/h
van 6.67 km/h
truck 7.58 km/h
bus 8.12 km/h
